# 02 — Fine-tuning Swin-Tiny 5 folds

GPU T4 direkomendasikan. Setiap fold dilatih hanya pada train, checkpoint terbaik dipilih lewat val loss, dan test diukur setelah pemilihan checkpoint.

**Pipeline revisi untuk artikel ilmiah.** Hasil tidak otomatis identik dengan skripsi 2026 karena split, kontrol, dan beberapa transformasi diperbarui. Jalankan notebook menurut urutan nomor.

In [ ]:
%pip -q install "timm>=1.0,<2" "torchvision>=0.20" scikit-learn

In [ ]:
from google.colab import drive
from pathlib import Path
import sys, os

drive.mount('/content/drive')
PROJECT=Path(os.environ.get('FFE_PROJECT','/content/drive/MyDrive/tugas_akhir/FFE_Publication_2026'))
assert PROJECT.exists(), f'Folder proyek tidak ditemukan: {PROJECT}'
os.environ['FFE_ROOT']=str(PROJECT)
sys.path.insert(0,str(PROJECT/'02_notebooks'))
from ffe_core import ROOT, paths, seed_all, ensure_dataset_local, load_manifest, load_split
P=paths(); seed_all()
print('Project:',ROOT)

In [ ]:
import torch, timm, numpy as np,pandas as pd, copy,json
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from ffe_core import ImageDataset,get_transform,make_model,metric_row,save_prediction
from tqdm.auto import tqdm
ensure_dataset_local()
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device',DEVICE,'timm',timm.__version__,'torch',torch.__version__)
EPOCHS=30; PATIENCE=7; BATCH_SIZE=16; LR=1e-4; WEIGHT_DECAY=1e-2
FOLDS=[1,2,3,4,5]; RETRAIN=False

## 1. Training functions (AMP bila CUDA)

In [ ]:
def make_loaders(fold):
 tr,val,te=[load_split(fold,k) for k in ('train','val','test')]
 def loader(d,training=False):
  return DataLoader(ImageDataset(d,get_transform(training)),batch_size=BATCH_SIZE,
                    shuffle=training,num_workers=2,pin_memory=(DEVICE.type=='cuda'))
 return tr,val,te,loader(tr,True),loader(val),loader(te)

def one_pass(model,loader,loss_fn,opt=None,scaler=None):
 train=opt is not None
 model.train(train)
 accum=0.;yy=[];pp=[];probs=[]
 for x,y in loader:
  x,y=x.to(DEVICE),y.to(DEVICE)
  if train:opt.zero_grad(set_to_none=True)
  with torch.set_grad_enabled(train):
   with torch.autocast(device_type='cuda',enabled=(DEVICE.type=='cuda')):
    logits=model(x);loss=loss_fn(logits,y)
   if train:
    scaler.scale(loss).backward();scaler.step(opt);scaler.update()
  accum+=float(loss.item())*len(y)
  pr=torch.softmax(logits.detach().float(),dim=1).cpu().numpy()
  probs.append(pr);yy.extend(y.cpu().tolist());pp.extend(pr.argmax(1).tolist())
 return accum/len(loader.dataset),np.array(yy),np.array(pp),np.vstack(probs)

## 2. Lima fold: train / validation / test dan simpan prediksi out-of-fold

In [ ]:
for fold in FOLDS:
 print('=== Fold',fold,'===')
 seed_all(42+fold)
 tr,val,te,tl,vl,sl=make_loaders(fold)
 ckpt=P['checkpoints']/f'swin_fold_{fold:02d}_best.pt'
 model=make_model(pretrained=not (ckpt.exists() and not RETRAIN),num_classes=3).to(DEVICE)
 if RETRAIN or not ckpt.exists():
  weights=compute_class_weight('balanced',classes=np.array([0,1,2]),y=tr.label.to_numpy())
  loss_fn=torch.nn.CrossEntropyLoss(weight=torch.as_tensor(weights,dtype=torch.float32,device=DEVICE))
  opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
  scaler=torch.amp.GradScaler('cuda',enabled=(DEVICE.type=='cuda'))
  best=np.inf;wait=0;hist=[]
  for epoch in range(1,EPOCHS+1):
   trloss,_,trpred,_=one_pass(model,tl,loss_fn,opt,scaler)
   vl_loss,vl_y,vl_pred,_=one_pass(model,vl,loss_fn)
   hist.append({'fold':fold,'epoch':epoch,'train_loss':trloss,'val_loss':vl_loss,
                'val_macro_f1':f1_score(vl_y,vl_pred,average='macro')})
   print(f'  {epoch}: train={trloss:.4f} val={vl_loss:.4f} F1={hist[-1]["val_macro_f1"]:.4f}')
   if vl_loss<best-1e-5:
    best=vl_loss;wait=0
    torch.save({'state_dict':model.state_dict(),'fold':fold,'best_val_loss':best,
                'architecture':'swin_tiny_patch4_window7_224','label_order':['Highly Fresh','Fresh','Not Fresh']},ckpt)
   else:
    wait+=1
    if wait>=PATIENCE:break
  pd.DataFrame(hist).to_csv(P['logs']/f'swin_train_fold_{fold:02d}.csv',index=False)
 else:print('  Existing checkpoint reused:',ckpt)
 state=torch.load(ckpt,map_location='cpu',weights_only=False)
 model.load_state_dict(state['state_dict']);model.eval()
 loss_fn=torch.nn.CrossEntropyLoss()
 _,yt,yp,prob=one_pass(model,sl,loss_fn)
 assert np.array_equal(yt,te.label.to_numpy()),'Test order mismatch'
 save_prediction(P['metrics']/f'pred_swin_head_fold_{fold:02d}.csv',te,yp,prob,'swin_head',fold)
 print('  Test:',metric_row(yt,yp))
 del model
 if DEVICE.type=='cuda':torch.cuda.empty_cache()
print('Notebook 02 selesai.')